In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', '../requirements.txt', '-q'])
print('✓ All dependencies installed.')

# Mount Google Drive to save checkpoints (run in Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/brain_tumor_checkpoints'

# For local training:
CHECKPOINT_DIR = '../checkpoints'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Setup complete. Checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
# Mount Google Drive to save checkpoints (run in Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/brain_tumor_checkpoints'

# For local training:
CHECKPOINT_DIR = '../checkpoints'

import os, sys
sys.path.insert(0, '..')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Setup complete. Checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
import yaml
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import f1_score, recall_score

from src.data.dataset import BrainTumorDataset
from src.data.augmentation import get_train_transform, get_val_transform
from src.model.classifier import build_model, get_class_weights

with open('../config.yaml') as f:
    CFG = yaml.safe_load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cpu':
    print('WARNING: Training on CPU will be very slow. Use Google Colab for GPU access.')

## Load Datasets

In [ ]:
SEED    = CFG['dataset']['random_seed']
BATCH   = CFG['training']['phase1_batch_size']

torch.manual_seed(SEED)
np.random.seed(SEED)

train_dataset = BrainTumorDataset(
    root_dir='../data/Training',
    transform=get_train_transform(),
    index_file='../data/splits/train_index.csv',
)
val_dataset = BrainTumorDataset(
    root_dir='../data/Training',
    transform=get_val_transform(),
    index_file='../data/splits/val_index.csv',
)

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} images | Val: {len(val_dataset)} images')
print(f'Train class counts: {train_dataset.class_counts()}')
print(f'Val class counts:   {val_dataset.class_counts()}')

## Training Loop Helper

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    n = len(all_labels)
    acc    = sum(p == l for p, l in zip(all_preds, all_labels)) / n
    recall = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
    f1     = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
    return total_loss / n, acc, recall, f1


def save_checkpoint(model, optimizer, epoch, metrics, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
    }, path)

print('Training helpers defined.')

## Phase 1 — Train Classification Head Only (Base Frozen)

In [ ]:
model = build_model(pretrained=True).to(DEVICE)
model.freeze_base()
print(f'Phase 1 trainable parameters: {model.count_trainable_params():,}')

class_weights = get_class_weights(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                             lr=CFG['training']['phase1_lr'])

EPOCHS_1 = CFG['training']['phase1_epochs']
history_p1 = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_recall': [], 'val_f1': []}
best_f1, best_recall = 0, 0

for epoch in range(1, EPOCHS_1 + 1):
    t_loss, t_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    v_loss, v_acc, v_recall, v_f1 = val_epoch(model, val_loader, criterion, DEVICE)

    history_p1['train_loss'].append(t_loss)
    history_p1['val_loss'].append(v_loss)
    history_p1['val_acc'].append(v_acc)
    history_p1['val_recall'].append(v_recall)
    history_p1['val_f1'].append(v_f1)

    ckpt_path = f'{CHECKPOINT_DIR}/p1_epoch{epoch:02d}_f1{v_f1:.3f}.pt'
    save_checkpoint(model, optimizer, epoch, {'f1': v_f1, 'recall': v_recall}, ckpt_path)

    if v_f1 > best_f1:
        best_f1 = v_f1
        best_recall = v_recall
        torch.save(model.state_dict(), '../models/phase1_best.pt')

    print(f'[P1 Epoch {epoch:02d}/{EPOCHS_1}] '
          f'train_loss={t_loss:.4f} | val_acc={v_acc:.4f} | '
          f'recall={v_recall:.4f} | f1={v_f1:.4f}')

print(f'\nPhase 1 complete. Best F1={best_f1:.4f}, Recall={best_recall:.4f}')

## Phase 2 — Fine-Tune Last 2 Convolutional Blocks

In [ ]:
# Load best Phase 1 weights
model.load_state_dict(torch.load('../models/phase1_best.pt', map_location=DEVICE, weights_only=True))
model.unfreeze_top_blocks(num_blocks=2)
print(f'Phase 2 trainable parameters: {model.count_trainable_params():,}')

# Lower learning rate to avoid destroying pretrained features
optimizer2 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=CFG['training']['phase2_lr'])

EPOCHS_2 = CFG['training']['phase2_epochs']
history_p2 = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_recall': [], 'val_f1': []}
best_f1_p2 = best_f1  # continue from Phase 1 best

for epoch in range(1, EPOCHS_2 + 1):
    t_loss, t_acc = train_epoch(model, train_loader, optimizer2, criterion, DEVICE)
    v_loss, v_acc, v_recall, v_f1 = val_epoch(model, val_loader, criterion, DEVICE)

    history_p2['train_loss'].append(t_loss)
    history_p2['val_loss'].append(v_loss)
    history_p2['val_acc'].append(v_acc)
    history_p2['val_recall'].append(v_recall)
    history_p2['val_f1'].append(v_f1)

    ckpt_path = f'{CHECKPOINT_DIR}/p2_epoch{epoch:02d}_f1{v_f1:.3f}.pt'
    save_checkpoint(model, optimizer2, epoch, {'f1': v_f1, 'recall': v_recall}, ckpt_path)

    if v_f1 >= best_f1_p2:
        best_f1_p2 = v_f1
        torch.save(model.state_dict(), '../models/classifier_best.pt')
        print(f'  --> New best model saved! F1={v_f1:.4f}')

    print(f'[P2 Epoch {epoch:02d}/{EPOCHS_2}] '
          f'train_loss={t_loss:.4f} | val_acc={v_acc:.4f} | '
          f'recall={v_recall:.4f} | f1={v_f1:.4f}')

print(f'\nTraining complete. Final best F1={best_f1_p2:.4f}')
print('Best model saved to: models/classifier_best.pt')

## Training Curves

In [ ]:
# Combine Phase 1 + Phase 2 history for plotting
all_recall = history_p1['val_recall'] + history_p2['val_recall']
all_f1     = history_p1['val_f1']     + history_p2['val_f1']
all_acc    = history_p1['val_acc']    + history_p2['val_acc']
epochs     = list(range(1, len(all_recall) + 1))
phase_boundary = EPOCHS_1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves — All Epochs', fontsize=13, fontweight='bold')

axes[0].plot(epochs, all_recall, 'b-o', label='Recall (Sensitivity)', markersize=4)
axes[0].plot(epochs, all_f1,     'g-o', label='F1-Score', markersize=4)
axes[0].plot(epochs, all_acc,    'r-o', label='Accuracy', markersize=4)
axes[0].axvline(phase_boundary + 0.5, color='gray', linestyle='--', label='Phase 1 → 2')
axes[0].axhline(0.95, color='blue', linestyle=':', alpha=0.5, label='Recall target (95%)')
axes[0].axhline(0.90, color='green', linestyle=':', alpha=0.5, label='F1 target (90%)')
axes[0].set_ylim([0.5, 1.0])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Score')
axes[0].set_title('Validation Metrics')
axes[0].legend()

all_train_loss = history_p1['train_loss'] + history_p2['train_loss']
all_val_loss   = history_p1['val_loss']   + history_p2['val_loss']
axes[1].plot(epochs, all_train_loss, 'b-o', label='Train Loss', markersize=4)
axes[1].plot(epochs, all_val_loss,   'r-o', label='Val Loss', markersize=4)
axes[1].axvline(phase_boundary + 0.5, color='gray', linestyle='--', label='Phase 1 → 2')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Curves')
axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to docs/training_curves.png')